In [ ]:
#| default_exp game/web

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import json

In [ ]:
#| export
import logging

for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    file_handler = logging.FileHandler('HexServer.txt')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)


read_url("https://raw.githubusercontent.com/AnswerDotAI/MonsterUI/refs/heads/main/docs/llms-ctx.txt")

In [ ]:
show(Strong("I am strong text"))

In [ ]:
sakura_headers = [Link(href='https://cdn.jsdelivr.net/npm/sakura.css/css/sakura.css', rel='stylesheet', type='text/css')]

In [ ]:


app = FastHTML(hdrs = sakura_headers) # A FastHTML app, including the sakura CSS link in the headers
rt = app.route
server = JupyUvi(app) # Starts a server on port 8000 hosting the app
     


In [ ]:
#| export
from monsterui.all import *


The `apsw.ThreadingViolationError` happens because APSW (which fastlite uses under the hood) checks that a connection is only used from the thread that created it. Since Starlette/uvicorn runs your route handlers in a **thread pool**, multiple request threads can hit `globalStore.db` concurrently.

This isn't really an async/await issue — it's a **shared connection across threads** issue. Here are two clean fixes:

**Option 1: Threading Lock (simplest)**

Serialize all DB access with a lock:

```python
import threading
db_lock = threading.Lock()

def ensure_user(session):
    with db_lock:
        uid = session.get('uid')
        row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
        # ... rest of db work
```

Wrap every function that touches the DB in `with db_lock:`. Simple, and perfectly fine for a game with moderate concurrency.

**Option 2: Thread-local connections**

Give each thread its own connection:

```python
import threading
_local = threading.local()

def get_db():
    if not hasattr(_local, 'db'):
        _local.db = database('hex.db')  # each thread gets its own connection
    return _local.db
```

Then use `get_db()` instead of `globalStore.db` in your route handlers. This allows true concurrent reads (especially with WAL mode).

**Which to pick?**

- **Lock** is easier — just wrap existing code, no structural changes. Fine for low-to-moderate traffic.
- **Thread-local** is better if you want concurrent reads, but you need to be careful about write conflicts.

For either approach, also consider enabling WAL mode on your database for better concurrent read performance:

```python
globalStore.db.execute("PRAGMA journal_mode=WAL")
```

I'd start with the threading lock since it's a one-line change per function and your current code structure stays the same.

In [ ]:
#| export
if server: server.stop(); del server
app = FastHTML(hdrs = Theme.violet.headers())
rt = app.route
server = JupyUvi(app)

Is there an async await we should do with our database? How do we thread?

In [ ]:
#| export
def webMe(*c): return HTMX(*c, host='', app=app)

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

read_url("https://raw.githubusercontent.com/AnswerDotAI/MonsterUI/refs/heads/main/docs/apilist.txt")

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion ,  unique_windy_edge, Hex, HexBackground, HexWrapper,  HexTouchMap, HexButtonGroup
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed

In [ ]:
#| export
from HexMagic.game.data import GameBoard,Kingdom,TradeRoute,Terrain,StyleCSS, Hex, TerraDemo, Geology, DrainageBasins, CountryFlag, GameStorage, TerrainTemplate, Settlement, Piece

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User

In [ ]:
#| export
import logging

logging.basicConfig(
    filename='base.text',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("getting Started")


In [ ]:
??ChunkCover.zoom_region_fast

read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

!cat ../../HexMagic/plot/*.py

## Helpers

#| export
@patch
def html(self:Terrain,wrapper:HexWrapper = None)->str:
    grid = self.hexGrid
    if wrapper is None:
        wrapper = HexWrapper(callBack=HexWrapper.route())
    clearStyle = StyleCSS("HexClear",stroke="orange",fill="white",opacity=0.25)
    hover = StyleCSS("hover",fill="purple",cursor="pointer" )
            #hover = StyleCSS("hover",fill="#007fff",cursor="pointer" )
    clearStyle.customize(hover)
            
    for i, h in enumerate(grid.hexes):
        grid.hexes[i].style = clearStyle
        #grid.hexes[i].label = str(i)
    #self.colorMap()
    grid.builder.add_style(clearStyle)
    grid.update(wrapper=wrapper,layer_name="hexes")
    for i, l in enumerate(grid.builder.layers):
        logging.info(f"layer[{i}] is {l.name}")
    return grid.builder.xml()

## ActiveGame

In [ ]:
#| export
@dataclass
class ActiveGame:
    board: GameBoard
    cover: ChunkCover
    world_id: int
    country_id: int = 0  # 0 = world view, >0 = zoomed into that kingdom
    selected_piece: str = ""
    selected_settlement: str = ""


## Database

In [ ]:
#| export
globalStore = GameStorage("gameData/WebDebug.db")

In [ ]:
#| export
# Helper to ensure we have a user row
def ensure_user(session) -> int:
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    uid = session['userid']
    row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
    if not row:
        from datetime import datetime
        now = int(datetime.now().timestamp())
        globalStore.users.insert({
            'id': uid, 'username': f'player_{uid}', 'email': '', 'password': '',
            'created': now, 'sessionID': str(uid), 'activeWorld': 0
        })
    return uid



def new_game_page():
    templates = TerrainTemplate.maps  # {'bayArea': 'bayArea_map', ...}
    form = Form(
        Div(
            Label("Map Template", cls="label"),
            Select(
                *[Option(name, value=name) for name in sorted(templates.keys())],
                name="template_name", cls="select select-bordered w-full"
            ),
            cls="form-control"
        ),
        Div(
            Label("Kingdoms", cls="label"),
            Input(type="number", name="kingdoms", value="5", min="1", max="10",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Lakes", cls="label"),
            Input(type="number", name="lakes", value="1", min="0", max="5",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Hex Radius", cls="label"),
            Input(type="number", name="radius", value="25", min="10", max="40",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Button("Create World", type="submit", cls="btn btn-primary mt-4"),
        action="/create_world", method="post",
        cls="card bg-base-200 shadow-lg p-6 space-y-4 max-w-md mx-auto"
    )
    return Titled("New Game",
        Div(
            H3("Choose Your World", cls="text-2xl font-bold text-center mb-6"),
            form,
            cls="flex flex-col items-center p-12"
        )
    )




In [ ]:
#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


In [ ]:
_game_cache = {}
_cache_lock = threading.Lock()

@patch
def active_board(self: GameStorage, user_id) -> ActiveGame:
    with _cache_lock:
        if user_id in _game_cache:
            logging.info(f"active_board: cache hit for user {user_id}")
            return _game_cache[user_id]
    
    logging.info(f"active_board: cache miss, building for user {user_id}")
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]:
        return None
    
    world_id = row[0]
    try:
        # gameboard() now loads cover internally and reuses cover.basin
        board = self.gameboard(world_id)
        active = ActiveGame(board=board, cover=board.cover, world_id=world_id)
        
        with _cache_lock:
            _game_cache[user_id] = active
        
        return active
    except Exception as e:
        logging.error(f"active_board: FAILED: {e}", exc_info=True)
        return None


## Handlers

In [ ]:
#| export
@rt
def hex_clicked(session, hex_id: int):
    logging.info(f"hex_clicked: session keys={list(session.keys())}")
    uid = ensure_user(session)
    logging.info(f"hex_clicked: uid={uid}")
    try:
        active = globalStore.active_board(uid)
        if not active:
            return RedirectResponse('/', status_code=303)

        board = active.board
        terrain = board.terrain
        countries = terrain.fields.get("country")

        if countries is None or hex_id < 0 or hex_id >= len(countries):
            debug_msg = f"Invalid hex {hex_id}"
            return (
                P("Invalid hex", id="map"),
                Div(debug_msg, id="debug-panel", hx_swap_oob="true")
            )

        country_id = int(countries[hex_id])

        if country_id > 0:
            # Clicked a kingdom — zoom in
            k = next((k for k in board.kingdoms if k.countryId == country_id), None)
            debug_msg = f"Zooming into {k.countryName if k else f'Kingdom {country_id}'}"
            
            map_content = kingdom(session, country_id)
            return (
                map_content,
                Div(debug_msg, id="debug-panel", hx_swap_oob="true")
            )
        elif country_id == 0:
            debug_msg = f"Hex {hex_id} is unclaimed land"
        else:
            debug_msg = f"Hex {hex_id} is water/mountains"
        
        return (
            P(debug_msg, id="map"),
            Div(debug_msg, id="debug-panel", hx_swap_oob="true")
        )

    except Exception as e:
        logging.error(f"showMap FAILED: {type(e).__name__}: {e}", exc_info=True)
        return (
            Div(P(f"Error: {e}"), id="map"),
            Div(f"hex_clicked error: {type(e).__name__}: {e}", id="debug-panel", hx_swap_oob="true")
        )


## Common Routes

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


In [ ]:
@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

In [ ]:
_game_cache = {}
_cache_lock = threading.Lock()

def invalidate_cache(user_id):
    """Call this when creating a new game or the board changes."""
    with _cache_lock:
        _game_cache.pop(user_id, None)


In [ ]:
@rt
def left_panel(session):
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    board = active.board
    terrain = board.terrain
    current_radius = int(terrain.hexGrid.radius)

    kingdom_links = []
    for k in board.kingdoms:
        kingdom_links.append(
            A(k.countryName or f"Kingdom {k.countryId}",
              hx_get=f"/kingdom/{k.countryId}",
              hx_target="#map",
              cls="btn btn-ghost btn-sm justify-start",
              style=f"color: {k.flag.primary}; border-left: 3px solid {k.flag.comp};"
              
              )
        )

    return Div(
        H4("Kingdoms", cls="text-lg font-bold mb-4"),
        Div(*kingdom_links, id="left-panel-content", cls="flex flex-col gap-1"),
        Divider(),
        H4("Hex Size", cls="text-lg font-bold mb-2"),
        Div(
            Input(type="range", name="radius", id="radius-slider",
                  min="10", max="40", value=str(current_radius),
                  hx_get="/showMap", hx_target="#map",
                  hx_trigger="change", cls="uk-range w-full"),
            P(f"{current_radius}px", id="radius-label", cls=TextPresets.muted_sm),
            Script("""
                me('#radius-slider').on('input', ev => {
                    me('#radius-label').textContent = ev.target.value + 'px';
                });
            """),
            cls="space-y-2"
        ),
        Divider(),
        A("🌍 New Game", href="/new_game", cls="btn btn-outline btn-sm w-full"),
        id="left-panel"
    )


In [ ]:
@rt
def select_kingdom(session, country_id: int):
    """Handle kingdom hex button click from left panel."""
    return kingdom(session, int(country_id))


@rt
def left_panel(session):
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    board = active.board
    terrain = board.terrain
    current_radius = int(terrain.hexGrid.radius)

    # Build styled hex buttons for each kingdom
    kingdom_hexes = []
    kingdom_values = []
    for k in board.kingdoms:
        style = StyleCSS(f"k_{k.countryId}",
                         fill=k.flag.primary,
                         stroke=k.flag.comp,
                         stroke_width=2)
        h = Hex(radius=25, center=MapCord(0, 0), style=style,
                label=k.countryName or f"Kingdom {k.countryId}")
        kingdom_hexes.append(h)
        kingdom_values.append(k.countryId)

    kingdom_buttons = HexButtonGroup(
        kingdom_hexes,
        name="country_id",
        values=kingdom_values,
        hx_post="/select_kingdom",
        hx_target="#map",
        hx_swap="outerHTML",
        direction="column",
        height=45,
        font_size=11,
        char_width=7,
    )

    # Radius slider
    radius_control = Div(
        P("Hex Size", cls=TextPresets.bold_sm),
        Input(type="range", name="radius", id="radius-slider",
              min="10", max="40", value=str(current_radius),
              hx_get="/showMap", hx_target="#map",
              hx_trigger="change", cls="uk-range w-full"),
        P(f"{current_radius}px", id="radius-label", cls=TextPresets.muted_sm),
        Script("""
            me('#radius-slider').on('input', ev => {
                me('#radius-label').textContent = ev.target.value + 'px';
            });
        """),
        cls="space-y-2"
    )

    # Size hex background to fit content
    n = len(board.kingdoms)
    panel_h = max(400, 120 + n * 55 + 120)  # kingdoms + controls
    panel_w = 240

    panel_content = Div(
        P("Kingdoms", cls=TextPresets.bold_sm),
        kingdom_buttons,
        Divider(),
        radius_control,
        Divider(),
        A("🌍 New Game", href="/new_game", cls="btn btn-outline btn-sm"),
        cls="flex flex-col items-center gap-2 p-4",
        style="position: relative; z-index: 1;"
    )
    return panel_content

    return HexBackground(
        panel_content,
        size=MapSize(panel_w, panel_h),
        fill="#f4edb2",
        levels=2,
        shadow=True,
        id="left-panel"
    )


Lets see what parchment looks like. and I meant to write there should just be one HexBackground for the pannel

Can we have the links be the color from the countryFlag for the kingdoms

In [ ]:
@rt
def showMap(session, radius: int = 0):
    try:
        logging.info(f"showMap: entering")
        uid = ensure_user(session)
        active = globalStore.active_board(uid)
        if not active:
            return P("No active game", id="map")

        board = active.board
        terrain = board.terrain
        
        logging.info(f"showMap: board has cover={hasattr(board, 'cover')}, active.cover={active.cover is not None}")
        
        grid = terrain.hexGrid
        builder = grid.builder
        builder.layers = []

        if radius > 0:
            grid.adjustRadius(radius)

        terrain.colorMap()
        grid.update()
        terrain.compute_climate()

        
        terrain.terrainCream()

        builder.adjust("climates", terrain.dottedClimate())
        builder.adjust("settlement", board.settlementOverlay())
        builder.adjust("countries", board.countries_overlay())
        builder.adjust("water", active.cover.basin.draw_watersheds())
        builder.adjust("names", board.names_overlay())

        #wrapper = HexWrapper(callBack=HexWrapper.route())

        #wrapper = HexWrapper(callBack=lambda grid, index: {

        #map_svg = terrain.html(wrapper=wrapper)
        #logging.info(f"showMap: svg length={len(map_svg)}")
        # Check if hx-get appears in the SVG output
        #logging.info(f"showMap: 'hx-get' in svg: {'hx-get' in map_svg}")
        #logging.info(f"showMap: first 500 of svg with hx: {[line for line in map_svg.split(chr(10)) if 'hx-get' in line][:3]}")

        on_click = HexWrapper.route("/hex_clicked", "#map")
        

        return HexTouchMap(grid, on_click=on_click, id="map", cls="w-full h-full")

        #return Div(NotStr(map_svg), cls="w-full h-full")
    except Exception as e:
        logging.error(f"showMap FAILED: {type(e).__name__}: {e}", exc_info=True)
        return (
            Div(P(f"Error: {e}"), id="map"),
            Div(f"showMap error: {type(e).__name__}: {e}", id="debug-panel", hx_swap_oob="true")
        )


I was thinking of redoing showMap, but with a HexTouchMap

In [ ]:
@rt
def new_game(session):
    return new_game_page()


In [ ]:
@rt
def game(session):
    logging.info(f"game: session keys={list(session.keys())}")
    uid = ensure_user(session)
    logging.info(f"game: uid={uid}")
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    return Titled("Hex Game",
        Div(
            # Left sidebar
            Div(
                id="left-panel",
                hx_get="/left_panel",
                hx_trigger="load",
                hx_indicator="#spinner",
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            
            # Main area
            Div(
                # Spinner — hidden by default, shown during any request that targets it
                Loading(cls=(LoadingT.spinner, LoadingT.lg),
                        htmx_indicator=True, id="spinner"),
                Div(id="map", hx_get="/showMap", hx_trigger="load",
                    hx_indicator="#spinner",
                    cls="flex-1 overflow-auto"),
                Div(id="map-controls", cls="h-16 bg-base-300 p-2 flex-shrink-0"),
                cls="flex-1 flex flex-col items-center justify-center relative"
            ),
            
            # Right sidebar
            Div(
                H4("Debug", cls="text-sm font-bold mb-2"),
                Div(id="debug-panel", cls="text-sm"),
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            cls="flex h-screen"
        )
    )


In [ ]:
#| export
@rt
def index(session):
    
    uid = ensure_user(session)
    logging.info(f"index: uid={uid}")
    active = globalStore.active_board(uid)
    if active:
        return RedirectResponse('/game', status_code=303)
    return new_game_page()

dummySession= {'userid': 667256 }
webMe(index(dummySession))

In [ ]:
!tail -10 base.text

Any thoughts? this isn'y a normal file File: /tmp/ipykernel_2611/1116320735.py

In [ ]:
#| export
@rt
def kingdom_hex_clicked(session, hex_id: int, country_id: int):
    uid = ensure_user(session)
    logging.info(f"kingdom_hex_clicked: uid={uid}")
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    board = active.board
    terrain = board.terrain

    result = globalStore.kingdom_detail(active.world_id, country_id, active.cover)
    coarse_idx = result.mapper(hex_id)

    countries = terrain.fields.get("country")
    if countries is None or coarse_idx < 0 or coarse_idx >= len(countries):
        return RedirectResponse('/showMap', status_code=303)

    owner = int(countries[coarse_idx])

    if owner == country_id or owner <= 0:
        debug_msg = "Zooming out to world view"
        return (
            RedirectResponse('/showMap', status_code=303),
            Div(debug_msg, id="debug-panel", hx_swap_oob="true")
        )
    else:
        k = next((k for k in board.kingdoms if k.countryId == owner), None)
        debug_msg = f"Zooming into {k.countryName if k else f'Kingdom {owner}'}"
        map_content = kingdom(session, owner)
        return (
            map_content,
            Div(debug_msg, id="debug-panel", hx_swap_oob="true")
        )


In [ ]:
#| export
@rt("/kingdom/{id}")
def kingdom(session, id: int):
    uid = ensure_user(session)
    logging.info(f"kingdom: uid={uid}")
    active = globalStore.active_board(uid)
    if not active:
        logging.info(f"kingdom: not active")
        return RedirectResponse('/', status_code=303)

    board = active.board
    terrain = board.terrain

    result = globalStore.kingdom_detail(active.world_id, id, active.cover)

    zoomed = result.terrain
    zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
    zoomed.hexGrid.builder.layers = []
    zoomed.colorMap()
    zoomed.hexGrid.update()

    c2f = result.invert_mapper()

    builder = zoomed.hexGrid.builder
    
    zoomed.terrainCream()

    builder.adjust("borders", board.countries_overlay(zoomed, c2f))
    builder.adjust("names", board.names_overlay(zoomed, c2f))

    if result.basins:
        builder.adjust("water", result.basins.draw_watersheds())

    k = next((k for k in board.kingdoms if k.countryId == id), None)
    title = k.countryName if k else f"Kingdom {id}"
    logging.info(f"kingdom {title} for uid={uid} is {id}")

    on_click = lambda grid, index: {
        "hx-post": "/kingdom_hex_clicked",
        "hx-vals": json.dumps({"hex_id": index, "country_id": id}),
        "hx-target": "#map"
    }

    return Div(
        Div(
            A("← Back to World", hx_get="/showMap", hx_target="#map",
              cls="btn btn-sm btn-outline"),
            H4(title, cls="text-lg font-bold"),
            cls="flex items-center gap-4 p-2"
        ),
        HexTouchMap(zoomed.hexGrid, on_click=on_click, cls="w-full h-full"),
        id="map"
    )


I want to redo kingdom using HexTouchMap

did I do it correctly?

Can you write the full function

read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
# Get all users
users_df = pd.DataFrame(globalStore.users())
print("Users:")
print(users_df)

In [ ]:
dummySession= {'userid': 64801 }
#webMe(index(dummySession))

Can you create a new game for the dummy session. Maybe san franciso?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! Based on your code, `"bayArea"` looks like the San Francisco-area template. Try:

```python
globalStore.create_game(dummySession['userid'], template_name="bayArea")
```

That should create a new game for user `64801` with the default 5 kingdoms and 1 lake. Want to tweak any of those parameters?

In [ ]:
globalStore.create_game(dummySession['userid'], template_name="bayArea")

In [ ]:
webMe(left_panel(dummySession))

In [ ]:
webMe(hex_clicked(dummySession,500))

In [ ]:
??HexTouchMap



def HexTouchMap(grid, on_click=None, hover_fill="rgba(128,128,128,0.3)",
                cursor="pointer", touch_layer="touch", **kwargs):
    """Render a HexGrid as an interactive SVG with an optional touch layer.

    Args:
        grid: HexGrid to render
        on_click: Callable(grid, index) -> dict of HTML/HTMX attrs per hex
                  e.g. HexWrapper.route("/hex_clicked", "#map")
        hover_fill: Fill color on hover for touch hexes
        cursor: CSS cursor style on hover
        touch_layer: Name of the SVG <g> layer for touch targets
        hex_layer: Name of the SVG <g> layer for hex rendering
        **kwargs: HTML/HTMX attrs passed to the wrapping Div (e.g. id, cls)

    Returns:
        FastHTML Div containing the SVG
    """
    # Render the visual hex layer
    grid.update()

    # Add interactive touch layer if on_click provided
    if on_click is not None:
        # Touch style: transparent fill, hover highlight
        touch_style = StyleCSS("hex-touch", fill="transparent", stroke="none", cursor=cursor)
        hover_style = StyleCSS("_hover", fill=hover_fill, cursor=cursor)
        touch_style.customize(hover_style)
        grid.builder.add_style(touch_style)

        # Build touch polygons with per-hex callback attrs
        touch_body = ""
        for i, hex_obj in enumerate(grid.hexes):
            if i not in grid.invalidRegion:
                touch_hex = Hex(hex_obj.radius, hex_obj.center, touch_style, v=hex_obj.v)
                attrs = on_click(grid, i)
                touch_body += "\t" + touch_hex.svg(attrs) + "\n"

        grid.builder.adjust(touch_layer, touch_body)

        for layer in grid.builder.layers:
            print(f"Layer {layer.name} size{len(layer.body)}")

    return Div(NotStr(grid.builder.xml()), **kwargs)


In [ ]:
webMe(showMap(dummySession))

In [ ]:
#webMe(kingdom(dummySession,1))

There is no map or countries

In [ ]:
#webMe(index({'userid': 999999}))

I don't see hex_clicked working.

Still no

This used to work

Is it ok?

I got a 404 create_world not found

In [ ]:
for r in app.routes:
    print(r.path, getattr(r, 'methods', ''))


In [ ]:
!tail -100 base.text

Is there a way to have an hour glass or spinner while this computes

Still no detail

Any thoughts

Now there is no svg for the main map

In [ ]:
#!cat ../../HexMagic/weather.py

In [ ]:
server.stop()

Can you update showMap?

The countryhtml needs to be complete refactored and I think your /kingdom/{id} route idea is better. We don't have countrydetails anymore. but we do have things like
```python
with GeoStorageDebugger(keep_on_error=True) as dbg:
    dbg.server = GameStorage(custom_path=dbg.db_path)
    
    terrain = TerraDemo().bayArea_map()
    terrain.carve_to_ocean(num_lakes=1)
    terrain.hexGrid.adjustRadius(10)
    
    board = GameBoard(terrain, top_n=5)
    board.expand_kingdoms(max_rounds=50)
    
    # Save everything
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Bay Area")
    board.save(dbg.server, world_id=cover.ident)
    
    # Zoom into first kingdom
    k = board.kingdoms[0]
    result = dbg.server.kingdom_detail(cover.ident, k.countryId, terrain)
    
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    
    c2f = result.invert_mapper()


    borders = board.countries_overlay(result.terrain, c2f)
    names   = board.names_overlay(result.terrain, c2f)
    settle  = board.settlementOverlay(result.terrain, c2f)
    builder = result.terrain.hexGrid.builder
    builder.adjust("borders",borders)
    builder.adjust("names",names)
    builder.adjust("settle",settle)
    

    
    
    print(f"{k.countryName}: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
result.terrain.hexGrid.builder.show()
```

So there are two cases for kingdom_hex_clicked
1. we are inside the kindgom we are showing so we need to zoom out
2. we are outside the kingdom we are showing so we need to zoom in on the new kingdom selected.

So we need to redo hex_clicked now so it uses the global store and will zoom if a country is selected

So I think this brings us to layout. I am thinking of three panels - main where the map is and left and right side panels. Each of these main panels might have details below. the right side panel would have debug information so hexClick messages would go there. Does this make sense

1. I think the side panels should be fixed width, but collapsible
2. We could just put up the latest message. This is more to help me as I go
3. I think it should zoom and replace
4. We will figure out that as we go along. We are going to want to update the active_game if we have drilled down on a country and more things later about interface. I do imagine there will be a secion below the map that would toggle viewing settlements/ rivers / kind of thing

Lets wire up the debug messaging and update ActiveGame

Which is going to be cleaner? Will the user notice a difference?

Yes lets see how hex_clicked looks

Yes lets do the same for kingdom_hex_clicked

Should we have a route that is three panel thing. right now showmap is just one panel

I think 2

Lets do the game route and then the modified showMap

What is taking so long to render

The caching isn't going to work when we switch countries. Can we profile somehow

In [ ]:
import time

@patch
def active_board_timed(self: GameStorage, user_id):
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]: return None
    world_id = row[0]
    
    t0 = time.perf_counter()
    cover_result = self.load_cover(world_id)
    t1 = time.perf_counter()
    logging.info(f"TIMING load_cover: {t1-t0:.3f}s")
    
    cover = cover_result.data
    
    t2 = time.perf_counter()
    board = self.gameboard(world_id, cover.terrain)
    t3 = time.perf_counter()
    logging.info(f"TIMING gameboard: {t3-t2:.3f}s")
    
    return ActiveGame(board=board, cover=cover, world_id=world_id)


In [ ]:
import cProfile, pstats, io

def profile_active_board(user_id):
    pr = cProfile.Profile()
    pr.enable()
    
    # Invalidate cache first so it actually does the work
    invalidate_cache(user_id)
    active = globalStore.active_board(user_id)
    
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(30)
    print(s.getvalue())
    return active

profile_active_board(64801)


In [ ]:
import cProfile, pstats, io

def profile_active_board(user_id):
    pr = cProfile.Profile()
    pr.enable()
    
    # Invalidate cache first so it actually does the work
    invalidate_cache(user_id)
    active = globalStore.active_board(user_id)
    
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(30)
    print(s.getvalue())
    return active

profile_active_board(64801)


So here is the issue - watersheds from scratch are 0^2. Gameboard has a world which has a geology which computes drainage basin from scratch. we need to switch so that Gameboard uses a chunkCover


I think we need some similar profiling on computing kingdoms. I think we are using DrainageBasins when we should be having our better watershed algorithm

In [ ]:
import cProfile, pstats, io, time

def profile_kingdom(user_id, country_id):
    """Profile kingdom_detail to see if basins are recomputed."""
    active = globalStore.active_board(user_id)
    if not active:
        print("No active game")
        return
    
    board = active.board
    terrain = board.terrain
    
    pr = cProfile.Profile()
    pr.enable()
    
    result = globalStore.kingdom_detail(active.world_id, country_id, terrain)
    
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(30)
    print(s.getvalue())
    return result

def profile_showmap(user_id):
    """Profile the full showMap rendering pipeline."""
    active = globalStore.active_board(user_id)
    if not active:
        print("No active game")
        return
    
    board = active.board
    terrain = board.terrain
    grid = terrain.hexGrid
    builder = grid.builder
    
    t0 = time.perf_counter()
    terrain.colorMap()
    t1 = time.perf_counter()
    print(f"colorMap:        {t1-t0:.3f}s")
    
    grid.update()
    t2 = time.perf_counter()
    print(f"grid.update:     {t2-t1:.3f}s")
    
    terrain.compute_climate()
    t3 = time.perf_counter()
    print(f"compute_climate: {t3-t2:.3f}s")
    
    builder.layers = []
    terrain.terrainCream()
    t4 = time.perf_counter()
    print(f"terrainCream:    {t4-t3:.3f}s")
    
    builder.adjust("climates", terrain.dottedClimate())
    t5 = time.perf_counter()
    print(f"dottedClimate:   {t5-t4:.3f}s")
    
    builder.adjust("settlement", board.settlementOverlay())
    t6 = time.perf_counter()
    print(f"settlement:      {t6-t5:.3f}s")
    
    builder.adjust("countries", board.countries_overlay())
    t7 = time.perf_counter()
    print(f"countries:       {t7-t6:.3f}s")
    
    builder.adjust("water", board.cover.basin.draw_watersheds())
    t8 = time.perf_counter()
    print(f"watersheds:      {t8-t7:.3f}s")
    
    wrapper = HexWrapper(callBack=HexWrapper.route())
    map_svg = terrain.html(wrapper=wrapper)
    t9 = time.perf_counter()
    print(f"html/svg:        {t9-t8:.3f}s")
    print(f"SVG size:        {len(map_svg):,} bytes")
    print(f"TOTAL:           {t9-t0:.3f}s")


In [ ]:
# Profile the world map rendering
profile_showmap(64801)


In [ ]:
# Profile zooming into kingdom 1
k = globalStore.active_board(64801).board.kingdoms[0]
print(f"Profiling kingdom: {k.countryName} (id={k.countryId})")
profile_kingdom(64801, k.countryId)


I refactored kingdom_detail. can you test with the new api

import cProfile, pstats, io, time
def profile_kingdom(user_id, country_id):
    """Profile kingdom_detail with new cover-based API."""
    active = globalStore.active_board(user_id)
    if not active:
        print("No active game")
        return
    
    pr = cProfile.Profile()
    pr.enable()
    
    result = globalStore.kingdom_detail(active.world_id, country_id, active.cover)
    
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(30)
    print(s.getvalue())
    return result

# Run it
k = globalStore.active_board(64801).board.kingdoms[0]
print(f"Profiling kingdom: {k.countryName} (id={k.countryId})")
profile_kingdom(64801, k.countryId)


redo

In [ ]:
import cProfile, pstats, io, time
def profile_kingdom(user_id, country_id):
    """Profile kingdom_detail with new cover-based API."""
    active = globalStore.active_board(user_id)
    if not active:
        print("No active game")
        return
    
    pr = cProfile.Profile()
    pr.enable()
    
    result = globalStore.kingdom_detail(active.world_id, country_id, active.cover)
    
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(30)
    print(s.getvalue())
    return result

# Run it
invalidate_cache(64801)
active = globalStore.active_board(64801)
k = active.board.kingdoms[0]
print(f"Profiling kingdom: {k.countryName} (id={k.countryId})")
profile_kingdom(64801, k.countryId)

In [ ]:
import cProfile, pstats, io, time
def profile_kingdom(user_id, country_id):
    """Profile kingdom_detail with new cover-based API."""
    active = globalStore.active_board(user_id)
    if not active:
        print("No active game")
        return
    
    pr = cProfile.Profile()
    pr.enable()
    
    result = globalStore.kingdom_detail(active.world_id, country_id, active.cover)
    
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(30)
    print(s.getvalue())
    return result

# Run it
invalidate_cache(64801)
active = globalStore.active_board(64801)
k = active.board.kingdoms[0]
print(f"Profiling kingdom: {k.countryName} (id={k.countryId})")
profile_kingdom(64801, k.countryId)

In [ ]:
server.stop()

decode cover?

Did things improve?


This seems like a great plan. My big lesson from all of this is drop to numpy as much as possible

Lets do the other speed ups later and focus in on db -> numpy -> terrain

Can you write this for me. I guess we will need the callback handler for the radio group of kingdoms